
# Analyze Lantern Second-Pass Candidates

Follow-up analysis for the local Second-Pass candidate list.

Main diagnostics:

- candidate \(i\)-band magnitude from `lsst_diaObject_i_scienceFluxMean`;
- per-alert science-image magnitudes from `lsst_diaSource_scienceFlux`;
- difference-image PSF S/N from `lsst_diaSource_psfFlux / psfFluxErr`;
- score correlations;
- sky, band, month, and alerts-per-locus distributions;
- Gaia motion/parallax/QSO information from the downloaded locus table;
- MILLiquas catalog-match flag;
- final candidate list after Gaia-based rejection.

MILLiquas and Gaia `in_qso_candidates` are **flags for inspection**, not rejection criteria.


In [ ]:

import os
import ast
import json
from collections.abc import Mapping

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------- INPUTS --------------------

CANDIDATES_FILE = "second_pass_output/second_pass_candidates.parquet"
CANDIDATE_ALERTS_FILE = "second_pass_output/second_pass_candidate_alerts.parquet"
LOCI_FILE = "./antares_data_clean_from_20260527/loci/loci_00000.parquet"

OUTPUT_DIR = "candidate_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Gaia rejection thresholds
PM_SIG_CUT = 3.0
PARALLAX_SIG_CUT = 3.0
GAIA_BP_RP_CUT = 2.5   # inactive unless Gaia color exists

AB_ZEROPOINT_NJY = 31.4

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
})


In [ ]:

# Load data

candidates = pd.read_parquet(CANDIDATES_FILE)
candidate_alerts = pd.read_parquet(CANDIDATE_ALERTS_FILE)
loci = pd.read_parquet(LOCI_FILE)

required_candidates = {"locus_id", "ra", "dec", "second_pass_score"}
required_alerts = {"locus_id", "alert_id", "mjd", "alert_properties"}

missing = required_candidates - set(candidates.columns)
if missing:
    raise KeyError(f"Missing candidate columns: {sorted(missing)}")

missing = required_alerts - set(candidate_alerts.columns)
if missing:
    raise KeyError(f"Missing alert columns: {sorted(missing)}")

print(f"Candidates:       {len(candidates):,}")
print(f"Candidate alerts: {len(candidate_alerts):,}")
print(f"Loci:             {len(loci):,}")


In [ ]:

# Helpers

def parse_value(value):
    if isinstance(value, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(value)
            except Exception:
                pass
    return value


def as_dict(value):
    value = parse_value(value)
    return value if isinstance(value, Mapping) else {}


def as_list(value):
    value = parse_value(value)

    if value is None:
        return []

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (list, tuple, set)):
        return list(value)

    if pd.isna(value):
        return []

    return [value]


def get_prop(value, key, default=np.nan):
    value = as_dict(value)
    result = value.get(key, default)
    return default if result is None else result


def normalize_band(series):
    s = series.astype(str)

    extracted = s.str.extract(
        r"band=['\"]([ugrizy])['\"]",
        expand=False,
    )
    plain = s.str.extract(
        r"^([ugrizy])$",
        expand=False,
    )

    return extracted.fillna(plain)


def flux_to_mag(flux):
    flux = np.asarray(flux, dtype=float)
    mag = np.full(flux.shape, np.nan)

    good = np.isfinite(flux) & (flux > 0)
    mag[good] = AB_ZEROPOINT_NJY - 2.5 * np.log10(flux[good])

    return mag


def catalog_has(value, token):
    token = token.lower()

    for item in as_list(value):
        if token in str(item).lower():
            return True

    return False


def numeric_list(value):
    vals = pd.to_numeric(
        pd.Series(as_list(value), dtype="object"),
        errors="coerce",
    ).dropna()

    return vals.to_numpy(dtype=float)


def bool_list(value):
    out = []

    for item in as_list(value):
        if isinstance(item, (bool, np.bool_)):
            out.append(bool(item))
        elif isinstance(item, str):
            if item.lower() == "true":
                out.append(True)
            elif item.lower() == "false":
                out.append(False)

    return out


## 1. Alert photometry

In [ ]:

# Extract alert-level quantities.

p = candidate_alerts["alert_properties"]

alerts = candidate_alerts[
    ["locus_id", "alert_id", "mjd"]
].copy()

alerts["band"] = p.map(
    lambda x: get_prop(x, "lsst_diaSource_band")
)

# Direct/science-image forced PSF flux
alerts["scienceFlux"] = p.map(
    lambda x: get_prop(x, "lsst_diaSource_scienceFlux")
)
alerts["scienceFluxErr"] = p.map(
    lambda x: get_prop(x, "lsst_diaSource_scienceFluxErr")
)

# Difference-image PSF flux
alerts["psfFlux"] = p.map(
    lambda x: get_prop(x, "lsst_diaSource_psfFlux")
)
alerts["psfFluxErr"] = p.map(
    lambda x: get_prop(x, "lsst_diaSource_psfFluxErr")
)

# Object-level weighted-mean i-band science flux carried by alerts
alerts["diaObject_i_scienceFluxMean"] = p.map(
    lambda x: get_prop(x, "lsst_diaObject_i_scienceFluxMean")
)

alerts["band"] = normalize_band(alerts["band"])

for col in [
    "mjd",
    "scienceFlux",
    "scienceFluxErr",
    "psfFlux",
    "psfFluxErr",
    "diaObject_i_scienceFluxMean",
]:
    alerts[col] = pd.to_numeric(alerts[col], errors="coerce")

alerts["science_mag"] = flux_to_mag(alerts["scienceFlux"])
alerts["diff_psf_snr"] = alerts["psfFlux"] / alerts["psfFluxErr"]
alerts["abs_diff_psf_snr"] = np.abs(alerts["diff_psf_snr"])

alerts["date"] = pd.to_datetime(
    alerts["mjd"],
    unit="D",
    origin=pd.Timestamp("1858-11-17"),
)
alerts["month"] = alerts["date"].dt.to_period("M").astype(str)

print(f"Valid scienceFlux magnitudes: {alerts['science_mag'].notna().sum():,}")
print(f"Valid difference PSF S/N:     {alerts['diff_psf_snr'].notna().sum():,}")
print(
    "Valid diaObject i scienceFluxMean rows: "
    f"{alerts['diaObject_i_scienceFluxMean'].notna().sum():,}"
)


In [ ]:

# Candidate-level summaries.
#
# diaObject_i_scienceFluxMean can evolve as the DiaObject is updated,
# so use the latest non-null value carried by the retained alerts.

def latest_nonnull(group, column):
    tmp = group[["mjd", column]].dropna(subset=[column]).sort_values("mjd")
    return tmp[column].iloc[-1] if len(tmp) else np.nan


rows = []

for locus_id, g in alerts.groupby("locus_id"):
    i_flux = latest_nonnull(g, "diaObject_i_scienceFluxMean")

    rows.append({
        "locus_id": locus_id,
        "n_alerts": len(g),
        "n_bands": g["band"].nunique(),
        "diaObject_i_scienceFluxMean": i_flux,
        "i_mag": flux_to_mag([i_flux])[0],
        "median_science_mag": g["science_mag"].median(),
        "median_abs_diff_psf_snr": g["abs_diff_psf_snr"].median(),
        "max_abs_diff_psf_snr": g["abs_diff_psf_snr"].max(),
        "first_mjd": g["mjd"].min(),
        "last_mjd": g["mjd"].max(),
    })

alert_summary = pd.DataFrame(rows)

analysis = candidates.merge(
    alert_summary,
    on="locus_id",
    how="left",
)


In [ ]:

# Primary candidate magnitude distribution:
# object-level weighted-mean i-band science flux.

plt.figure(figsize=(4, 3))
plt.hist(
    analysis["i_mag"].dropna(),
    bins=25,
    histtype="step",
)
plt.xlabel(r"$i$-band science magnitude")
plt.ylabel("Candidates")
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()

print(analysis["i_mag"].describe())


In [ ]:

# Per-alert science-image magnitude distribution by band.

plt.figure(figsize=(4, 3))

for band in "ugrizy":
    x = alerts.loc[
        alerts["band"] == band,
        "science_mag",
    ].dropna()

    if len(x):
        plt.hist(
            x,
            bins=25,
            histtype="step",
            label=band,
        )

plt.xlabel("DiaSource science magnitude")
plt.ylabel("Alerts")
plt.gca().invert_xaxis()
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Candidate i magnitude versus Second-Pass score.

good = (
    analysis["i_mag"].notna()
    & analysis["second_pass_score"].notna()
)

plt.figure(figsize=(4, 3))
plt.scatter(
    analysis.loc[good, "i_mag"],
    analysis.loc[good, "second_pass_score"],
    s=14,
    alpha=0.7,
)
plt.xlabel(r"$i$-band science magnitude")
plt.ylabel("Second-Pass score")
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()

if good.sum() > 1:
    r = np.corrcoef(
        analysis.loc[good, "i_mag"],
        analysis.loc[good, "second_pass_score"],
    )[0, 1]
    print(f"Pearson r = {r:.3f}")


In [ ]:

# Difference-image PSF S/N distribution.

plt.figure(figsize=(4, 3))

for band in "ugrizy":
    x = alerts.loc[
        alerts["band"] == band,
        "abs_diff_psf_snr",
    ].dropna()

    if len(x):
        plt.hist(
            x,
            bins=30,
            histtype="step",
            label=band,
        )

plt.xlabel(r"Difference PSF $|S/N|$")
plt.ylabel("Alerts")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Candidate median difference-image PSF S/N versus score.

good = (
    analysis["median_abs_diff_psf_snr"].notna()
    & analysis["second_pass_score"].notna()
)

plt.figure(figsize=(4, 3))
plt.scatter(
    analysis.loc[good, "median_abs_diff_psf_snr"],
    analysis.loc[good, "second_pass_score"],
    s=14,
    alpha=0.7,
)
plt.xlabel(r"Median difference PSF $|S/N|$")
plt.ylabel("Second-Pass score")
plt.tight_layout()
plt.show()


## 2. Sky, band, month, and alert counts

In [ ]:

# Candidate sky distribution.

plt.figure(figsize=(5, 3))
plt.scatter(
    analysis["ra"],
    analysis["dec"],
    s=12,
    alpha=0.7,
)
plt.xlabel("RA [deg]")
plt.ylabel("Dec [deg]")
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()


In [ ]:

# Band distribution.

band_counts = (
    alerts["band"]
    .value_counts()
    .reindex(list("ugrizy"))
    .fillna(0)
)

plt.figure(figsize=(4, 3))
plt.bar(band_counts.index, band_counts.values)
plt.xlabel("Band")
plt.ylabel("Alerts")
plt.tight_layout()
plt.show()

band_counts


In [ ]:

# Alert sky distribution by band.

sky_alerts = alerts.merge(
    candidates[["locus_id", "ra", "dec"]],
    on="locus_id",
    how="left",
)

plt.figure(figsize=(5, 3))

for band in "ugrizy":
    tmp = sky_alerts[sky_alerts["band"] == band]

    if len(tmp):
        plt.scatter(
            tmp["ra"],
            tmp["dec"],
            s=8,
            alpha=0.5,
            label=band,
        )

plt.xlabel("RA [deg]")
plt.ylabel("Dec [deg]")
plt.gca().invert_xaxis()
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# Alert counts by month.

month_counts = (
    alerts["month"]
    .value_counts()
    .sort_index()
)

x = np.arange(len(month_counts))

plt.figure(figsize=(5, 3))
plt.hist(
    np.repeat(x, month_counts.values.astype(int)),
    bins=np.arange(len(month_counts) + 1) - 0.5,
    histtype="step",
)
plt.xlabel("Month")
plt.ylabel("Alerts")
plt.xticks(
    x,
    month_counts.index,
    rotation=45,
    ha="right",
)
plt.tight_layout()
plt.show()

month_counts


In [ ]:

# Alert month on the sky.

month_labels = sorted(alerts["month"].dropna().unique())
month_map = {month: i for i, month in enumerate(month_labels)}

tmp = sky_alerts.dropna(
    subset=["ra", "dec", "month"]
).copy()
tmp["month_code"] = tmp["month"].map(month_map)

plt.figure(figsize=(5, 3))
sc = plt.scatter(
    tmp["ra"],
    tmp["dec"],
    c=tmp["month_code"],
    s=8,
    alpha=0.6,
)

cbar = plt.colorbar(sc)
cbar.set_ticks(np.arange(len(month_labels)))
cbar.set_ticklabels(month_labels)
cbar.set_label("Alert month")

plt.xlabel("RA [deg]")
plt.ylabel("Dec [deg]")
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()


In [ ]:

# Number of alerts per locus.

plt.figure(figsize=(4, 3))
plt.hist(
    analysis["n_alerts"].dropna(),
    bins=25,
    histtype="step",
)
plt.xlabel("Alerts per locus")
plt.ylabel("Candidates")
plt.tight_layout()
plt.show()

analysis["n_alerts"].describe()


In [ ]:

# Alerts per locus on the sky.

tmp = analysis.dropna(
    subset=["ra", "dec", "n_alerts"]
)

plt.figure(figsize=(5, 3))
sc = plt.scatter(
    tmp["ra"],
    tmp["dec"],
    c=tmp["n_alerts"],
    s=14,
    alpha=0.7,
)

cbar = plt.colorbar(sc)
cbar.set_label("Alerts per locus")

plt.xlabel("RA [deg]")
plt.ylabel("Dec [deg]")
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()


## 3. Gaia and MILLiquas flags

In [ ]:

# Merge downloaded ANTARES locus information.

locus_info = loci[
    loci["locus_id"].isin(analysis["locus_id"])
].copy()

analysis = analysis.merge(
    locus_info,
    on="locus_id",
    how="left",
    suffixes=("", "_locus"),
)

expected_gaia = [
    "gaia_parallax_over_error",
    "gaia_pmdec",
    "gaia_pmdec_error",
    "gaia_pmra",
    "gaia_pmra_error",
    "gaia_in_qso_candidates",
    "gaia_phot_g_mean_mag",
    "gaia_phot_g_mean_flux_over_error",
]

print("Downloaded Gaia columns found:")
for col in expected_gaia:
    print(f"  {col}: {col in analysis.columns}")

print(f"catalogs column: {'catalogs' in analysis.columns}")


In [ ]:

# Summarize the Gaia arrays saved by query_v3-2.ipynb.
#
# Each locus can have multiple Gaia matches. The arrays are aligned by Gaia
# catalog object, so compute the significance for each match and then store
# the maximum significance as a conservative stellar-contaminant flag.

def gaia_row_summary(row):
    par = numeric_list(row.get("gaia_parallax_over_error"))
    pmra = numeric_list(row.get("gaia_pmra"))
    pmra_err = numeric_list(row.get("gaia_pmra_error"))
    pmdec = numeric_list(row.get("gaia_pmdec"))
    pmdec_err = numeric_list(row.get("gaia_pmdec_error"))
    gmag = numeric_list(row.get("gaia_phot_g_mean_mag"))
    qso = bool_list(row.get("gaia_in_qso_candidates"))

    n_pm = min(len(pmra), len(pmra_err), len(pmdec), len(pmdec_err))

    pm_sig = []

    for i in range(n_pm):
        if pmra_err[i] > 0 and pmdec_err[i] > 0:
            pm_sig.append(
                np.sqrt(
                    (pmra[i] / pmra_err[i])**2
                    + (pmdec[i] / pmdec_err[i])**2
                )
            )

    return pd.Series({
        "gaia_n_matches": max(
            len(par),
            len(pmra),
            len(gmag),
            len(qso),
        ),
        "gaia_pm_sig_max": np.nanmax(pm_sig) if len(pm_sig) else np.nan,
        "gaia_parallax_sig_max": np.nanmax(np.abs(par)) if len(par) else np.nan,
        "gaia_phot_g_mean_mag_brightest": np.nanmin(gmag) if len(gmag) else np.nan,
        "gaia_in_qso_candidates_any": any(qso) if len(qso) else False,
    })


gaia_summary = analysis.apply(
    gaia_row_summary,
    axis=1,
)

analysis = pd.concat(
    [analysis, gaia_summary],
    axis=1,
)

analysis["gaia_match"] = (
    analysis["gaia_n_matches"] > 0
)

analysis["milliquas_match"] = (
    analysis["catalogs"].map(
        lambda x: catalog_has(x, "milliquas")
    )
    if "catalogs" in analysis.columns
    else False
)

print(f"Gaia matches:                  {analysis['gaia_match'].sum():,}")
print(
    "Gaia in_qso_candidates=True: "
    f"{analysis['gaia_in_qso_candidates_any'].sum():,}"
)
print(f"MILLiquas matches:             {analysis['milliquas_match'].sum():,}")


## 4. Gaia rejection

In [ ]:

analysis["high_pm_flag"] = (
    analysis["gaia_pm_sig_max"] > PM_SIG_CUT
)

analysis["high_parallax_flag"] = (
    analysis["gaia_parallax_sig_max"] > PARALLAX_SIG_CUT
)

# query_v3-2.ipynb did not save Gaia colors.
# Keep the column for future datasets, but do not reject on color here.
analysis["very_red_flag"] = False

analysis["reject_gaia_flag"] = (
    analysis["high_pm_flag"]
    | analysis["high_parallax_flag"]
    | analysis["very_red_flag"]
)

analysis["keep_flag"] = ~analysis["reject_gaia_flag"]

print(f"PM significance > {PM_SIG_CUT}:       {analysis['high_pm_flag'].sum():,}")
print(f"Parallax significance > {PARALLAX_SIG_CUT}: {analysis['high_parallax_flag'].sum():,}")
print("Gaia BP-RP cut:                   inactive (not downloaded)")
print(f"Rejected by Gaia cuts:            {analysis['reject_gaia_flag'].sum():,}")
print(f"Candidates retained:              {analysis['keep_flag'].sum():,}")


In [ ]:

# Gaia G magnitude is useful as a diagnostic even though it is not a rejection cut.

plt.figure(figsize=(4, 3))
plt.hist(
    analysis["gaia_phot_g_mean_mag_brightest"].dropna(),
    bins=25,
    histtype="step",
)
plt.xlabel("Gaia G magnitude")
plt.ylabel("Candidates")
plt.gca().invert_xaxis()
plt.tight_layout()
plt.show()


In [ ]:

# Gaia motion/parallax diagnostics.

plt.figure(figsize=(4, 3))
plt.hist(
    analysis["gaia_pm_sig_max"].dropna(),
    bins=25,
    histtype="step",
)
plt.axvline(PM_SIG_CUT, ls="--")
plt.xlabel("Maximum Gaia proper-motion significance")
plt.ylabel("Candidates")
plt.tight_layout()
plt.show()


plt.figure(figsize=(4, 3))
plt.hist(
    analysis["gaia_parallax_sig_max"].dropna(),
    bins=25,
    histtype="step",
)
plt.axvline(PARALLAX_SIG_CUT, ls="--")
plt.xlabel("Maximum Gaia |parallax / error|")
plt.ylabel("Candidates")
plt.tight_layout()
plt.show()


## 5. Final lists

In [ ]:

def rejection_reason(row):
    reasons = []

    if row["high_pm_flag"]:
        reasons.append("high_pm")

    if row["high_parallax_flag"]:
        reasons.append("high_parallax")

    if row["very_red_flag"]:
        reasons.append("very_red")

    return ";".join(reasons)


analysis["reject_reason"] = analysis.apply(
    rejection_reason,
    axis=1,
)

wanted_columns = [
    # Identity / classifier
    "locus_id",
    "ra",
    "dec",
    "second_pass_score",
    "first_cut_max_score",
    "n_tagged",
    "n_detections",
    "n_lsst_alerts",
    "percent_tagged",

    # Photometry / alerts
    "n_alerts",
    "n_bands",
    "diaObject_i_scienceFluxMean",
    "i_mag",
    "median_science_mag",
    "median_abs_diff_psf_snr",
    "max_abs_diff_psf_snr",
    "first_mjd",
    "last_mjd",

    # Gaia / MILLiquas
    "gaia_match",
    "gaia_n_matches",
    "gaia_pm_sig_max",
    "gaia_parallax_sig_max",
    "gaia_phot_g_mean_mag_brightest",
    "gaia_in_qso_candidates_any",
    "milliquas_match",

    # Selection
    "high_pm_flag",
    "high_parallax_flag",
    "very_red_flag",
    "reject_reason",
    "keep_flag",
]

output_columns = [
    col for col in wanted_columns
    if col in analysis.columns
]

full_list = (
    analysis[output_columns]
    .sort_values(
        "second_pass_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

filtered_list = (
    full_list[full_list["keep_flag"]]
    .sort_values(
        [
            "milliquas_match",
            "gaia_in_qso_candidates_any",
            "second_pass_score",
        ],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

print(f"Original candidates:               {len(full_list):,}")
print(f"Filtered candidates:               {len(filtered_list):,}")
print(
    "MILLiquas matches retained:        "
    f"{filtered_list['milliquas_match'].sum():,}"
)
print(
    "Gaia QSO-candidate flags retained: "
    f"{filtered_list['gaia_in_qso_candidates_any'].sum():,}"
)

filtered_list.head(20)


In [ ]:

# Save two logical catalogs, each in CSV and Parquet format.

files = {
    "full_csv": os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates_with_flags.csv",
    ),
    "full_parquet": os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates_with_flags.parquet",
    ),
    "filtered_csv": os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates_filtered.csv",
    ),
    "filtered_parquet": os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates_filtered.parquet",
    ),
}

full_list.to_csv(files["full_csv"], index=False)
full_list.to_parquet(files["full_parquet"], index=False)

filtered_list.to_csv(files["filtered_csv"], index=False)
filtered_list.to_parquet(files["filtered_parquet"], index=False)

print("Saved:")
for path in files.values():
    print(" ", path)


In [ ]:
print(
    "Both MILLiquas and Gaia QSO:",
    (
        filtered_list["milliquas_match"]
        & filtered_list["gaia_in_qso_candidates_any"]
    ).sum()
)

print(
    "Neither:",
    (
        ~filtered_list["milliquas_match"]
        & ~filtered_list["gaia_in_qso_candidates_any"]
    ).sum()
)


### Output files

There are only **two logical catalogs**; each is saved in two formats.

**`second_pass_candidates_with_flags`**  
All Second-Pass candidates. It keeps the classifier score, photometric summaries,
Gaia/MILLiquas information, rejection flags, rejection reason, and `keep_flag`.

**`second_pass_candidates_filtered`**  
Only rows with `keep_flag=True`, i.e. candidates not rejected by the Gaia proper-motion
or parallax cuts. MILLiquas and Gaia-QSO flags are retained and used only to sort useful
matches toward the top.

`.csv` is convenient for quick inspection; `.parquet` preserves data types and is better
for later Python analysis.
